# 🐝 말벌 AI 시험 보기

## 사용법 — 코드는 안 고쳐도 돼요
1. 구글 드라이브 `hornet_finetune` 폴더 안에 **`test`** 폴더 만들기
2. 거기에 **새로 찍은 영상이나 사진** 넣기 (공부할 때 안 쓴 것!)
   - 말벌 표본 영상 + 빈 종이 영상 + (있으면) 다른 곤충 영상
3. 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU → 저장**
4. 위 메뉴 **런타임 → 모두 실행** → 드라이브 연결 창이 뜨면 **허용**
5. 결과는 맨 아래 화면 + 드라이브 `hornet_finetune/test_results` 폴더 (네모 쳐진 영상)

❗ 빨간 에러가 나면 캡처해서 Claude에게 보내 주세요.

## 1. 준비 (드라이브 연결, 파일 확인)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
DRIVE_DIR = '/content/drive/MyDrive/hornet_finetune'
MODEL     = f'{DRIVE_DIR}/best_hornet_v1.pt'   # 공부를 마친 AI
TEST_DIR  = f'{DRIVE_DIR}/test'                # 시험 볼 영상·사진
OUT_DIR   = f'{DRIVE_DIR}/test_results'        # 결과가 저장될 곳
CONF      = 0.5                                # AI가 50% 이상 확신할 때만 '말벌'로 인정

VID_EXT = {'.mp4', '.mov', '.avi', '.mkv', '.wmv'}
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}
if not os.path.exists(MODEL):
    raise FileNotFoundError('❌ hornet_finetune 폴더에 best_hornet_v1.pt가 없어요.')
os.makedirs(TEST_DIR, exist_ok=True)
files = sorted(f for f in glob.glob(f'{TEST_DIR}/*') if os.path.splitext(f)[1].lower() in VID_EXT | IMG_EXT)
if not files:
    raise FileNotFoundError('❌ hornet_finetune/test 폴더에 영상이나 사진이 없어요. 넣고 다시 [런타임 → 모두 실행] 하세요.')
print('시험 볼 파일:')
for f in files: print('  ', os.path.basename(f))

## 2. 프로그램 설치

In [ ]:
%cd /content
!test -d yolov5 || git clone -q https://github.com/ultralytics/yolov5
%cd /content/yolov5
!pip install -q -r requirements.txt
# 최신 버전끼리 안 맞는 그림 저장 부분 고치기 (빨간 에러 방지)
!sed -i 's/annotator.im.save(fname)/Image.fromarray(np.asarray(annotator.im)).save(fname)/' utils/plots.py

## 3. 시험 보기 (영상이 길면 몇 분 걸려요)

In [ ]:
import shutil
shutil.rmtree(OUT_DIR, ignore_errors=True)
!python detect.py --weights {MODEL} --source {TEST_DIR} --conf-thres {CONF} --img 640 \
    --save-txt --project {DRIVE_DIR} --name test_results --exist-ok

## 4. 성적표
- **말벌 영상**이면 → '말벌 찾은 장면'이 **높을수록** 좋아요
- **빈 종이·다른 곤충 영상**이면 → **0%에 가까울수록** 좋아요 (아니면 헛것을 본 것)

In [ ]:
import cv2, torch, pathlib
import sys; sys.path.insert(0, '/content/yolov5')
ck = torch.load(MODEL, map_location='cpu', weights_only=False)
names = ck['model'].names
names = names if isinstance(names, dict) else dict(enumerate(names))
HORNET = [k for k, v in names.items() if v == 'velutina']
lab_dir = pathlib.Path(OUT_DIR) / 'labels'

def has_hornet(txt):
    if not txt.exists(): return False
    return any(int(l.split()[0]) in HORNET for l in txt.read_text().strip().splitlines() if l.strip())

print(f"{'파일':30s} {'전체 장면':>8s} {'말벌 찾은 장면':>14s}")
print('-' * 60)
for f in files:
    p = pathlib.Path(f)
    if p.suffix.lower() in VID_EXT:
        total = int(cv2.VideoCapture(f).get(cv2.CAP_PROP_FRAME_COUNT))
        found = sum(has_hornet(lab_dir / f'{p.stem}_{i}.txt') for i in range(1, total + 1))
    else:
        total, found = 1, int(has_hornet(lab_dir / f'{p.stem}.txt'))
    pct = 100 * found / total if total else 0
    print(f'{p.name[:30]:30s} {total:8d} {found:9d} ({pct:5.1f}%)')

## 5. 결과 눈으로 보기 (빨간 네모 = AI가 찾은 것)

In [ ]:
import matplotlib.pyplot as plt
shots = []
for f in files:
    out = pathlib.Path(OUT_DIR) / pathlib.Path(f).name
    if not out.exists():
        cand = list(pathlib.Path(OUT_DIR).glob(pathlib.Path(f).stem + '.*'))
        if not cand: continue
        out = cand[0]
    if out.suffix.lower() in VID_EXT:
        cap = cv2.VideoCapture(str(out)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        for k in range(4):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * (k + 0.5) / 4)); ok, fr = cap.read()
            if ok: shots.append((f'{out.name} ({k+1}/4)', fr))
    else:
        shots.append((out.name, cv2.imread(str(out))))
for i in range(0, len(shots), 4):
    row = shots[i:i+4]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax in axes: ax.axis('off')
    for ax, (t, im) in zip(axes, row):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax.set_title(t[:35], fontsize=8)
    plt.tight_layout(); plt.show()
print('✅ 끝! 네모 쳐진 영상은 드라이브 hornet_finetune/test_results 폴더에 있어요.')